# DA6401 — Fix: Retrain Localizer (dataset bbox bug fixed)

**BEFORE RUNNING:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Click **Save Version** → **Save & Run All (Commit)** (NOT Quick Save!)

**What this notebook does:**
- Downloads existing classifier.pth from Google Drive (pretrained encoder)
- Retrains the localizer for 80 epochs using FIXED dataset (min_visibility=0.01)
  - Old bug: min_visibility=0.3 caused bboxes to be dropped → full-image fallback → bad IoU
  - Fix is already in the repo (commit 2e3e78b)
- Reports final IoU score

**Expected improvement:**
- Before fix: Acc@IoU=0.5 ≈ 20%
- After fix: Acc@IoU=0.5 ≈ 60%+

**Output:**
- `checkpoints/localizer.pth` — retrained localizer
- Download from Output tab → upload to Google Drive → update LOCALIZER_DRIVE_ID in models/multitask.py

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

# Clone latest code (contains the min_visibility=0.01 fix)
!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -3
%cd /kaggle/working/repo
!git log --oneline -5
!pip install -q wandb albumentations gdown scikit-learn

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
print(f"GPU: {gpu}")
assert torch.cuda.is_available(), "NO GPU — go to Settings and enable GPU T4 x2!"

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")
print("Dataset ready.")

# Verify the min_visibility fix is present
with open('data/pets_dataset.py') as f:
    content = f.read()
if 'min_visibility=0.01' in content:
    print("✓ Dataset fix confirmed: min_visibility=0.01")
else:
    print("✗ WARNING: Old min_visibility=0.3 still present — wrong branch!")

In [ ]:
# Download existing classifier.pth from Drive (pretrained encoder for localizer)
import gdown
CLS_DRIVE_ID = "1jPrAaVV3XYwfuX4y3Z1y2gXnGUYqHXJJ"
cls_path = f"{CKPT}/classifier.pth"
print("Downloading classifier.pth (pretrained encoder for localizer)...")
gdown.download(id=CLS_DRIVE_ID, output=cls_path, quiet=False, fuzzy=True)
if os.path.exists(cls_path):
    size_mb = os.path.getsize(cls_path) / 1e6
    print(f"✓ classifier.pth downloaded ({size_mb:.1f} MB)")
else:
    print("✗ Download failed — localizer will train without pretrained encoder")

## 1. Retrain Localizer (80 epochs, fixed dataset)

The key fix: `min_visibility=0.01` in BboxParams means RandomResizedCrop no longer drops bboxes.
Previously bboxes were dropped (min_visibility=0.3) → model trained on full-image fallback bbox → poor IoU.

In [ ]:
# Retrain localizer with fixed dataset
!python train.py --task localize --experiment fix-localizer \
    --epochs 80 --lr 1e-3 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

print("\n--- Localizer retraining complete → localizer.pth ---")

## 2. Evaluate Results

In [ ]:
import os, torch
from data.pets_dataset import create_dataloaders
from models.localization import VGG11Localizer
from train import compute_iou

CKPT = "/kaggle/working/checkpoints"
device = torch.device("cuda")

_, val_loader, test_loader = create_dataloaders(
    root="./data/oxford_pet", batch_size=64, num_workers=2
)

loc_path = f"{CKPT}/localizer.pth"
model = VGG11Localizer().to(device)
model.load_state_dict(torch.load(loc_path, map_location=device, weights_only=False))
model.eval()

print("="*60)
print("  LOCALIZER EVALUATION")
print("="*60)

for split_name, loader in [("val", val_loader), ("test", test_loader)]:
    total_iou, iou_50_count, total = 0.0, 0, 0
    with torch.no_grad():
        for b in loader:
            pred = model(b["image"].to(device)).cpu()
            ious = compute_iou(pred, b["bbox"])
            total_iou += ious.sum().item()
            iou_50_count += (ious >= 0.5).sum().item()
            total += len(ious)
    print(f"  {split_name}: mean IoU = {total_iou/total:.4f}, Acc@IoU=0.5 = {iou_50_count/total*100:.1f}%")

print("\n  Target: Acc@IoU=0.5 >= 60%")
print("\n  Files:")
for f in sorted(os.listdir(CKPT)):
    size_mb = os.path.getsize(f"{CKPT}/{f}") / 1e6
    print(f"    {f:40s} {size_mb:6.1f} MB")

print("\n  ↑ Download localizer.pth from the Output tab")
print("  ↑ Upload to Google Drive → share → get file ID")
print("  ↑ Update LOCALIZER_DRIVE_ID in models/multitask.py")
print("="*60)

## 3. (Optional) Classifier Continued Training

Run this ONLY if localizer evaluation above shows Acc@IoU≥60%.

**WARNING**: This changes encoder weights → may affect segmentation Dice.
Only useful if classification F1 is still below 0.8 after resubmission.

Run this cell only if you want to try improving classification further.

In [ ]:
# ── OPTIONAL: Continue classifier training for 40 more epochs (lr=1e-4) ──
# WARNING: This changes encoder weights. If you run this:
#   1. Upload the new classifier.pth to Drive
#   2. ALSO need to retrain U-Net segmentation (encoder changed)
#   3. Much safer to skip this and just use the localizer fix

# Uncomment to run:
# !python train.py --task classify --experiment fix-classifier --dropout 0.5 \
#     --epochs 40 --lr 1e-4 --batch-size 32 --num-workers 2 \
#     --checkpoint-dir {CKPT}
# !cp {CKPT}/classifier.pth {CKPT}/classifier_continued.pth
# print("\n--- Continued classifier → classifier_continued.pth ---")

print("(Optional step — not run automatically)")
print("Uncomment the above lines if you want to try classifier improvement.")